In [1]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib as mpl
import pandas as pd
from matplotlib import animation

from tools.sportec_data import SportecData

pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 30)

%matplotlib inline
mpl.rcParams['animation.embed_limit'] = 100

In [2]:
# It can take about one or two minutes to parse tracking data using kloppy
match_id = "J03WMX"
match = SportecData(match_id)
match.lineup.head()

Loading the tracking data...
Transforming the tracking data coordinates...


,team_id,team_name,home_away,player_id,uniform_number,object_id,player_name,starting,playing_position,captain
0,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-0027KL,2,away_2,Dayot Upamecano,True,RCB,False
1,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-J017RE,4,away_4,M. de Ligt,True,LCB,False
2,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-0027G0,5,away_5,Benjamin Pavard,True,RB,False
3,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-0002F5,6,away_6,Joshua Kimmich,True,RDM,False
4,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-0027G6,7,away_7,Serge Gnabry,True,CF,False


In [11]:
from autoevent.poss import PossessionDetector

In [12]:
detector = PossessionDetector(match.tracking)

In [14]:
result = detector.run()

In [15]:
print(result[result["is_loss"] | result["is_gain"]][
    [
        "frame_id",
        "ball_control",
        "controller_id",
        "control_sequence_id",
        "is_loss",
        "loss_player",
        "is_gain",
        "gain_player",
    ]
])

        frame_id ball_control controller_id control_sequence_id  is_loss loss_player  is_gain gain_player
0          10000   possession        away_7                   1     True      away_7    False        <NA>
35         10035   possession        away_2                   2    False        <NA>     True      away_2
40         10040   possession        away_2                   2     True      away_2    False        <NA>
57         10057   possession       home_11                   3     True     home_11    False        <NA>
81         10081   possession       away_25                   4     True     away_25    False        <NA>
...          ...          ...           ...                 ...      ...         ...      ...         ...
145478    174770   possession        away_2                2306     True      away_2    False        <NA>
145489    174781   possession        away_5                2307     True      away_5    False        <NA>
145508    174800   possession       away_10   

In [17]:
from tools.animator import Animator

events = result[result['is_gain'] | result['is_loss']].copy()
events['event_type'] = events.apply(lambda row: 'gain' if row['is_gain'] else 'loss', axis=1)
events['player_id'] = events.apply(lambda row: row['gain_player'] if row['is_gain'] else row['loss_player'], axis=1)

track_dict = {'main': match.tracking}
animator = Animator(track_dict=track_dict, events=events, show_events=True, play_speed=2)
anim = animator.run(max_frames=500, fps=12)
anim.save('possession_events.mp4', writer='ffmpeg', fps=12)
anim